# Experiment Tracking — Examples

This notebook demonstrates recording model parameters, validation metrics, environment information, and artifacts. It uses a synthetic dataset and can run without an external data download.

Optional MLflow cells use a local tracking store. Do not log secrets or sensitive records.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import hashlib

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)
import joblib

RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)

## 1. Create a synthetic dataset

In [ ]:
X_array, y = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    weights=[0.6, 0.4],
    random_state=RANDOM_STATE,
)
feature_names = [f"feature_{i}" for i in range(1, X_array.shape[1] + 1)]
X = pd.DataFrame(X_array, columns=feature_names)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_holdout, y_holdout, test_size=0.50, random_state=RANDOM_STATE, stratify=y_holdout
)
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

## 2. Train and track multiple configurations

In [ ]:
records = []
models = {}

for c_value in [0.1, 1.0, 10.0]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=c_value, max_iter=1000, random_state=RANDOM_STATE)),
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    run_name = f"logistic-regression-C-{c_value}"
    models[run_name] = model
    records.append({
        "run_name": run_name,
        "model": "LogisticRegression",
        "C": c_value,
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "python_version": platform.python_version(),
        "sklearn_version": sklearn.__version__,
        "data_version": "synthetic-v1",
        "git_commit": "unknown",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    })

results = pd.DataFrame(records).sort_values("f1", ascending=False).reset_index(drop=True)
results

## 3. Save experiment results

In [ ]:
results_path = ARTIFACT_DIR / "experiment_results.csv"
results.to_csv(results_path, index=False)
print("Saved:", results_path.resolve())

## 4. Save the best validation run and metadata

In [ ]:
best_run = results.iloc[0]
best_model = models[best_run["run_name"]]
best_val_predictions = best_model.predict(X_val)

model_path = ARTIFACT_DIR / "best_model.joblib"
joblib.dump(best_model, model_path)  # Only load artifacts from trusted sources.

report_path = ARTIFACT_DIR / "classification_report.txt"
report_path.write_text(
    classification_report(y_val, best_val_predictions, zero_division=0),
    encoding="utf-8",
)

metadata = {
    "run_name": str(best_run["run_name"]),
    "model_version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "parameters": {
        "C": float(best_run["C"]),
        "max_iter": int(best_run["max_iter"]),
        "random_state": int(best_run["random_state"]),
    },
    "validation_metrics": {
        metric: float(best_run[metric])
        for metric in ["accuracy", "precision", "recall", "f1"]
    },
    "features": feature_names,
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "data_version": "synthetic-v1",
    "git_commit": "unknown",
    "artifact": model_path.name,
}
metadata_path = ARTIFACT_DIR / "best_run.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(json.dumps(metadata, indent=2))

## 5. Evaluate the selected model once on the test split

In [ ]:
test_predictions = best_model.predict(X_test)
test_metrics = {
    "test_accuracy": float(accuracy_score(y_test, test_predictions)),
    "test_precision": float(precision_score(y_test, test_predictions, zero_division=0)),
    "test_recall": float(recall_score(y_test, test_predictions, zero_division=0)),
    "test_f1": float(f1_score(y_test, test_predictions, zero_division=0)),
}
test_metrics

## 6. Smoke test the persisted artifact

In [ ]:
loaded_model = joblib.load(model_path)  # Only load files from a trusted source.
loaded_predictions = loaded_model.predict(X_test)
assert np.array_equal(test_predictions, loaded_predictions)
assert len(loaded_predictions) == len(X_test)
print("Artifact smoke test passed.")

## 7. Optional: log a run with MLflow

In [ ]:
# Optional dependency: pip install mlflow
# Run this cell if MLflow is installed in your environment.
try:
    import mlflow

    mlflow.set_experiment("experiment-tracking-practice")
    with mlflow.start_run(run_name=str(best_run["run_name"])):
        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("C", float(best_run["C"]))
        mlflow.log_param("max_iter", int(best_run["max_iter"]))
        mlflow.log_metric("validation_accuracy", float(best_run["accuracy"]))
        mlflow.log_metric("validation_f1", float(best_run["f1"]))
        mlflow.log_metric("test_accuracy", test_metrics["test_accuracy"])
        mlflow.log_artifact(str(metadata_path))
    print("Logged run to MLflow.")
except ImportError:
    print("MLflow is not installed. Install it with: python -m pip install mlflow")

## 8. Calculate an artifact checksum

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("SHA-256:", sha256_file(model_path))

## Key takeaways

- Record parameters and metrics for every run.
- Compare configurations on the same validation split.
- Use the test set for a final evaluation after model selection.
- Save metadata and the artifact together.
- Do not log secrets or sensitive records.
- Never load untrusted pickle/joblib artifacts.